# 03. Training: Multi-Label "Partially Clouded" Classification

This notebook trains the second-level classifier for the hierarchical
system's most difficult category: **partially clouded** grains, which can
exhibit up to four co-occurring traits (Shinpaku, Base White, Back White,
Belly White) simultaneously. Because a grain can have more than one trait at
once, this is framed as multi-label classification (independent sigmoid
outputs) rather than single-label softmax classification.

Pipeline (see `src/sake_rice_inspection/{folds,datasets,models,training,experiment,reporting}.py`):
1. **Fold construction** — collect grain crops (single- and composite-trait),
   derive a multi-hot label per crop, and write a stratified k-fold CSV.
2. **Training** — for each fold: train a fresh classification head, then
   unfreeze and fine-tune the whole backbone, with flip-augmentation on the
   training split and early stopping on macro-F1.
3. **Evaluation & reporting** — exact-match / Hamming accuracy / macro-F1
   per fold, confusion matrices, and a consolidated Excel export of every
   validation prediction.

> **Note on data**: this notebook expects grain crops produced by
> `01_Preprocessing_Cellpose.ipynb` (optionally with edible-rice crops
> adapted by `02_Domain_Adaptation.ipynb`). The original NDA-protected
> dataset is not included in this repository.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("..") / "src"))

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from sake_rice_inspection.folds import (
    collect_composite_trait_samples,
    collect_single_trait_samples,
    filter_by_minimum_class_count,
    make_multilabel_folds_csv,
    read_folds_csv,
)
from sake_rice_inspection.datasets import MultiLabelGrainDataset
from sake_rice_inspection.models import build_model
from sake_rice_inspection.training import (
    compute_pos_weight,
    eval_metrics_multilabel,
    labels_to_str,
    train_one_epoch,
)
from sake_rice_inspection.visualization import plot_curves, plot_multiclass_confusion, plot_multilabel_metrics
from sake_rice_inspection.experiment import create_experiment_dir, save_experiment_readme, summarize_fold_results, update_readme_with_results
from sake_rice_inspection.reporting import create_unified_prediction_excel

## 1. Build the fold CSV

Single-trait crops get a one-hot label from their folder name; composite
crops get a multi-hot label decoded from their combination-folder name.
Classes with fewer samples than the number of folds are dropped, since
`StratifiedKFold` requires every class to appear in every split.

In [ ]:
LABEL_COLUMNS = ["心白", "基白", "背白", "腹白"]  # Shinpaku, Base White, Back White, Belly White
FOLDS = 4
SEED = 42

CROP_DIR = "../outputs/crops"
CROP_DIR_COMPOSITE = "../outputs/crops_composite"
FOLDS_CSV = "../outputs/folds_part_white.csv"

samples = (
    collect_single_trait_samples(CROP_DIR, allowed_classes=LABEL_COLUMNS, label_columns=LABEL_COLUMNS)
    + collect_composite_trait_samples(CROP_DIR_COMPOSITE, label_columns=LABEL_COLUMNS)
)
samples = filter_by_minimum_class_count(samples, min_count=FOLDS)
print(f"Samples after filtering: {len(samples)}")

make_multilabel_folds_csv(samples, FOLDS_CSV, label_columns=LABEL_COLUMNS, folds=FOLDS, seed=SEED)
all_rows, class_names = read_folds_csv(FOLDS_CSV)
print(f"Classes: {class_names}")

## 2. Configure and start an experiment

Every run gets a timestamped directory with a README recording its
hyperparameters up front (see `save_experiment_readme`); results are
appended once training finishes.

In [ ]:
ARCH = "resnet101"  # any of sake_rice_inspection.models.SUPPORTED_ARCHITECTURES
IMG_SIZE = 224
SCALE = 1.0
BATCH_SIZE = 32
EPOCHS_HEAD = 8
EPOCHS_FT = 20
LR_HEAD = 3e-4
LR_FT = 1e-4
EARLY_STOP_PATIENCE = 5

exp_root = create_experiment_dir(f"../outputs/train_part_white/{ARCH}_analysis", "crops_cv_4fold")

save_experiment_readme(exp_root, {
    "tag": "crops_cv_4fold",
    "description": "Hierarchical classification, layer 2-3: multi-label partially-clouded traits.",
    "crop_dir": f"{CROP_DIR} + {CROP_DIR_COMPOSITE}",
    "folds_csv": FOLDS_CSV,
    "num_classes": len(class_names),
    "details": "Sigmoid multi-label classification over Shinpaku/Base White/Back White/Belly White.",
    "class_names": class_names,
    "folds": FOLDS,
    "arch": ARCH,
    "img_size": IMG_SIZE,
    "batch_size": BATCH_SIZE,
    "epochs_head": EPOCHS_HEAD,
    "lr_head": LR_HEAD,
    "epochs_ft": EPOCHS_FT,
    "lr_ft": LR_FT,
    "seed": SEED,
    "augmentation": "4x (original, horizontal flip, vertical flip, both), training split only",
    "augmentation_details": "see datasets.apply_flip_augmentation",
    "scale": SCALE,
})

## 3. Cross-validation loop

For each fold: train a frozen-backbone head, then unfreeze for fine-tuning
with early stopping on validation macro-F1. `BCEWithLogitsLoss` with a
per-label `pos_weight` (see `compute_pos_weight`) accounts for label
imbalance across the four traits.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
n_classes = len(class_names)
all_predictions = []
fold_results = []

for fold in range(FOLDS):
    print(f"\n=== Fold {fold} ===")

    tr_items = [(p, lab) for (p, lab, lstr, f) in all_rows if f != fold]
    va_items = [(p, lab, lstr) for (p, lab, lstr, f) in all_rows if f == fold]
    va_items_for_training = [(p, lab) for (p, lab, lstr) in va_items]

    tr_ds = MultiLabelGrainDataset(tr_items, scale=SCALE, img_size=IMG_SIZE, augment=True)
    va_ds = MultiLabelGrainDataset(va_items_for_training, scale=SCALE, img_size=IMG_SIZE, augment=False)
    tr_dl = DataLoader(tr_ds, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True)
    va_dl = DataLoader(va_ds, batch_size=BATCH_SIZE, shuffle=False, pin_memory=True)

    pos_weight = compute_pos_weight(np.array([lab for _, lab in tr_items]), device)

    model = build_model(ARCH, n_classes, freeze_backbone=True).to(device)
    optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LR_HEAD)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    run_dir = Path(exp_root) / f"fold_{fold}"
    run_dir.mkdir(exist_ok=True)
    best_path = run_dir / "best_model.pt"
    history = {"epoch": [], "train_loss": [], "val_acc": [], "val_f1": []}
    best_f1 = -1.0

    for ep in range(1, EPOCHS_HEAD + 1):
        tr_loss = train_one_epoch(model, tr_dl, device, optimizer, criterion)
        _, hamming_acc, macro_f1, *_ = eval_metrics_multilabel(model, va_dl, device)
        history["epoch"].append(len(history["epoch"]) + 1)
        history["train_loss"].append(float(tr_loss))
        history["val_acc"].append(float(hamming_acc))
        history["val_f1"].append(float(macro_f1))
        if macro_f1 > best_f1:
            best_f1 = macro_f1
            torch.save(model.state_dict(), best_path)

    for p in model.parameters():
        p.requires_grad = True
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR_FT)
    wait = 0
    for ep in range(1, EPOCHS_FT + 1):
        tr_loss = train_one_epoch(model, tr_dl, device, optimizer, criterion)
        _, hamming_acc, macro_f1, *_ = eval_metrics_multilabel(model, va_dl, device)
        history["epoch"].append(len(history["epoch"]) + 1)
        history["train_loss"].append(float(tr_loss))
        history["val_acc"].append(float(hamming_acc))
        history["val_f1"].append(float(macro_f1))
        if macro_f1 > best_f1:
            best_f1 = macro_f1
            torch.save(model.state_dict(), best_path)
            wait = 0
        else:
            wait += 1
            if wait >= EARLY_STOP_PATIENCE:
                print("Early stop.")
                break

    model.load_state_dict(torch.load(best_path, map_location=device))
    exact_match, hamming_acc, macro_f1, y_true, y_pred, y_prob = eval_metrics_multilabel(model, va_dl, device)

    plot_curves(history, save_path=str(run_dir / "curves.png"))
    plot_multilabel_metrics(y_true, y_pred, class_names, save_path=str(run_dir / "confusion_multilabel.png"))
    y_true_str = [lstr for (_, _, lstr) in va_items]
    y_pred_str = labels_to_str(y_pred, class_names)
    plot_multiclass_confusion(y_true_str, y_pred_str, save_path=str(run_dir / "confusion_multiclass.png"))

    fold_results.append({
        "fold": fold,
        "eval_count": len(y_true),
        "exact_match_count": int((y_true == y_pred).all(axis=1).sum()),
        "exact_match_acc": float(exact_match),
        "hamming_acc": float(hamming_acc),
        "macro_f1": float(macro_f1),
    })
    print(f"Fold {fold} done. exact_match={exact_match:.4f} hamming_acc={hamming_acc:.4f} macro_f1={macro_f1:.4f}")

    model.eval()
    for img_path, true_labels, label_str in va_items:
        x, _ = MultiLabelGrainDataset([(img_path, true_labels)], scale=SCALE, img_size=IMG_SIZE)[0]
        with torch.no_grad():
            probs = torch.sigmoid(model(x.unsqueeze(0).to(device))).cpu().numpy()[0]
        all_predictions.append({"fold": fold, "img_path": img_path, "true_labels": true_labels, "true_label_str": label_str, "probs": probs})

## 4. Aggregate results and export

The final README section records overall exact-match / Hamming / macro-F1
across all folds; `all_predictions.xlsx` contains a thumbnail-annotated row
for every validation grain across every fold.

In [ ]:
summary = summarize_fold_results(fold_results)
update_readme_with_results(exp_root, summary)
print(f"Exact match acc: {summary['exact_match_acc']:.4f}")
print(f"Hamming acc:     {summary['hamming_acc']:.4f}")
print(f"Macro F1:        {summary['macro_f1']:.4f}")

excel_path = create_unified_prediction_excel(all_predictions, class_names, exp_root, image_root=CROP_DIR)
print(f"Predictions written to: {excel_path}")